<a href="https://colab.research.google.com/github/Duncainniza/test/blob/main/Sichande_Duncain_SoilMoisture_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improved Soil Moisture (`sm_surface`) Prediction

**Improvements over baseline XGBoost model:**
1. Extended lag features up to 30 days (vs. 3 days baseline)
2. Cyclical calendar encodings (sin/cos of day-of-year, month)
3. Richer rolling & EWM features for both target and precipitation
4. Ensemble of GradientBoosting + ExtraTrees + RandomForest (50/25/25 weighting)
5. Recursive multi-step forecasting with rolling EWM updates
6. More estimators (800 GBR trees) and tuned hyperparameters


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')


In [5]:
# Uncomment if running in Google Colab
from google.colab import drive
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Assignment 4'

df_train  = pd.read_csv(f'{base_path}/training_data.csv')
df_train['date'] = pd.to_datetime(df_train['date'])
df_train  = df_train.sort_values('date').reset_index(drop=True)

df_precip = pd.read_csv(f'{base_path}/precipitation_data.csv')
df_precip['date'] = pd.to_datetime(df_precip['date'])

df_test   = pd.read_csv(f'{base_path}/test_data_X.csv')
df_test['date'] = pd.to_datetime(df_test['date'])
df_test   = df_test.sort_values('date').reset_index(drop=True)

# Merge precipitation
df_train  = df_train.merge(df_precip, on='date', how='left').fillna({'precipitation_mm': 0})
df_test   = df_test.merge(df_precip,  on='date', how='left').fillna({'precipitation_mm': 0})

print(f'Training period: {df_train["date"].min()} → {df_train["date"].max()}')
print(f'Test period:     {df_test["date"].min()} → {df_test["date"].max()}')
print(f'Train shape: {df_train.shape}, Test shape: {df_test.shape}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training period: 2023-01-01 00:00:00 → 2024-09-30 00:00:00
Test period:     2024-10-01 00:00:00 → 2025-01-01 00:00:00
Train shape: (639, 3), Test shape: (93, 2)


# 2. Feature Engineering


In [6]:
TARGET_LAGS  = [1, 2, 3, 5, 7, 14, 21, 30]  # much richer than baseline's 3
PRECIP_LAGS  = [1, 2, 3, 5, 7, 14]
ROLL_WINDOWS = [3, 7, 14, 30]

FEAT_COLS = (
    ['precipitation_mm', 'month', 'dayofyear', 'week', 'season',
     'sin_doy', 'cos_doy', 'sin_month', 'cos_month']
    + [f'precip_lag{l}'  for l in PRECIP_LAGS]
    + ['precip_roll3', 'precip_roll7', 'precip_roll14', 'precip_roll30',
       'precip_ewm7', 'precip_ewm14', 'precip_cum_30']
    + [f'target_lag{l}'  for l in TARGET_LAGS]
    + ['target_roll3', 'target_roll7', 'target_roll14', 'target_roll30',
       'target_ewm7', 'target_ewm14']
)
print(f'Total features: {len(FEAT_COLS)}')
print(FEAT_COLS)


Total features: 36
['precipitation_mm', 'month', 'dayofyear', 'week', 'season', 'sin_doy', 'cos_doy', 'sin_month', 'cos_month', 'precip_lag1', 'precip_lag2', 'precip_lag3', 'precip_lag5', 'precip_lag7', 'precip_lag14', 'precip_roll3', 'precip_roll7', 'precip_roll14', 'precip_roll30', 'precip_ewm7', 'precip_ewm14', 'precip_cum_30', 'target_lag1', 'target_lag2', 'target_lag3', 'target_lag5', 'target_lag7', 'target_lag14', 'target_lag21', 'target_lag30', 'target_roll3', 'target_roll7', 'target_roll14', 'target_roll30', 'target_ewm7', 'target_ewm14']


In [7]:
# --- Build training feature matrix ---
df_tr = df_train.copy()

# Cyclical calendar features
df_tr['month']     = df_tr['date'].dt.month
df_tr['dayofyear'] = df_tr['date'].dt.dayofyear
df_tr['week']      = df_tr['date'].dt.isocalendar().week.astype(int)
df_tr['season']    = (df_tr['month'] % 12 + 3) // 3
df_tr['sin_doy']   = np.sin(2 * np.pi * df_tr['dayofyear'] / 365)
df_tr['cos_doy']   = np.cos(2 * np.pi * df_tr['dayofyear'] / 365)
df_tr['sin_month'] = np.sin(2 * np.pi * df_tr['month'] / 12)
df_tr['cos_month'] = np.cos(2 * np.pi * df_tr['month'] / 12)

# Precipitation features
for lag in PRECIP_LAGS:
    df_tr[f'precip_lag{lag}'] = df_tr['precipitation_mm'].shift(lag).fillna(0)
for win in ROLL_WINDOWS:
    df_tr[f'precip_roll{win}'] = df_tr['precipitation_mm'].rolling(win, min_periods=1).mean()
df_tr['precip_ewm7']   = df_tr['precipitation_mm'].ewm(span=7,  adjust=False).mean()
df_tr['precip_ewm14']  = df_tr['precipitation_mm'].ewm(span=14, adjust=False).mean()
df_tr['precip_cum_30'] = df_tr['precipitation_mm'].rolling(30, min_periods=1).sum()

# Target lag/rolling features
for lag in TARGET_LAGS:
    df_tr[f'target_lag{lag}'] = df_tr['sm_surface'].shift(lag)
for win in ROLL_WINDOWS:
    df_tr[f'target_roll{win}'] = df_tr['sm_surface'].rolling(win, min_periods=1).mean()
df_tr['target_ewm7']  = df_tr['sm_surface'].ewm(span=7,  adjust=False).mean()
df_tr['target_ewm14'] = df_tr['sm_surface'].ewm(span=14, adjust=False).mean()

df_tr   = df_tr.dropna().reset_index(drop=True)
X_train = df_tr[FEAT_COLS].values
y_train = df_tr['sm_surface'].values
print(f'Training feature matrix: {X_train.shape}')


Training feature matrix: (609, 36)


# 3. Cross-Validation (Time Series Split)


In [8]:
tscv = TimeSeriesSplit(n_splits=5)

gbr_cv = GradientBoostingRegressor(
    n_estimators=800, learning_rate=0.02, max_depth=4,
    subsample=0.8, min_samples_leaf=5, max_features=0.8, random_state=42
)
cv_mses = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    gbr_cv.fit(X_train[tr_idx], y_train[tr_idx])
    pred = gbr_cv.predict(X_train[val_idx])
    mse  = mean_squared_error(y_train[val_idx], pred)
    cv_mses.append(mse)
    print(f'  Fold {fold+1}: MSE = {mse:.6f}')

print(f'\nMean CV MSE: {np.mean(cv_mses):.6f}')


  Fold 1: MSE = 0.000281
  Fold 2: MSE = 0.000080
  Fold 3: MSE = 0.000058
  Fold 4: MSE = 0.000056
  Fold 5: MSE = 0.000028

Mean CV MSE: 0.000100


# 4. Train Ensemble on Full Training Data


In [9]:
# GradientBoosting — best single model
gbr = GradientBoostingRegressor(
    n_estimators=800, learning_rate=0.02, max_depth=4,
    subsample=0.8, min_samples_leaf=5, max_features=0.8, random_state=42
)
gbr.fit(X_train, y_train)
print('GradientBoostingRegressor trained')

# ExtraTreesRegressor — captures local variance well
etr = ExtraTreesRegressor(
    n_estimators=500, max_depth=8, min_samples_leaf=5,
    max_features=0.8, random_state=42, n_jobs=-1
)
etr.fit(X_train, y_train)
print('ExtraTreesRegressor trained')

# RandomForestRegressor — stable bagging baseline
rfr = RandomForestRegressor(
    n_estimators=500, max_depth=8, min_samples_leaf=5,
    max_features=0.8, random_state=42, n_jobs=-1
)
rfr.fit(X_train, y_train)
print('RandomForestRegressor trained')

# In-sample MSE (sanity check)
train_pred = 0.5*gbr.predict(X_train) + 0.25*etr.predict(X_train) + 0.25*rfr.predict(X_train)
print(f'\nEnsemble in-sample MSE: {mean_squared_error(y_train, train_pred):.6f}')


GradientBoostingRegressor trained
ExtraTreesRegressor trained
RandomForestRegressor trained

Ensemble in-sample MSE: 0.000009


# 5. Recursive Multi-Step Prediction


In [10]:
# Build running buffers from training history
sm_history = list(df_train['sm_surface'].values)
all_precip = list(df_train['precipitation_mm'].values) + list(df_test['precipitation_mm'].values)
n_train    = len(df_train)

test_predictions = []

for i, (date, precip) in enumerate(zip(df_test['date'], df_test['precipitation_mm'])):
    row     = {}
    cur_idx = n_train + i
    d       = pd.Timestamp(date)

    # Calendar
    row['precipitation_mm'] = precip
    row['month']     = d.month
    row['dayofyear'] = d.dayofyear
    row['week']      = d.isocalendar()[1]
    row['season']    = (d.month % 12 + 3) // 3
    row['sin_doy']   = np.sin(2 * np.pi * d.dayofyear / 365)
    row['cos_doy']   = np.cos(2 * np.pi * d.dayofyear / 365)
    row['sin_month'] = np.sin(2 * np.pi * d.month / 12)
    row['cos_month'] = np.cos(2 * np.pi * d.month / 12)

    # Precipitation lags / rolls (fully known)
    for lag in PRECIP_LAGS:
        idx = cur_idx - lag
        row[f'precip_lag{lag}'] = all_precip[idx] if idx >= 0 else 0.0
    for win in ROLL_WINDOWS:
        s = max(0, cur_idx - win)
        row[f'precip_roll{win}'] = np.mean(all_precip[s:cur_idx]) if cur_idx > 0 else 0.0
    rp = all_precip[:cur_idx]
    row['precip_ewm7']  = pd.Series(rp).ewm(span=7,  adjust=False).mean().iloc[-1] if rp else 0
    row['precip_ewm14'] = pd.Series(rp).ewm(span=14, adjust=False).mean().iloc[-1] if rp else 0
    row['precip_cum_30']= sum(all_precip[max(0, cur_idx-30):cur_idx])

    # Target lags / rolls — updated with predictions
    for lag in TARGET_LAGS:
        idx = len(sm_history) - lag
        row[f'target_lag{lag}'] = sm_history[idx] if idx >= 0 else sm_history[0]
    for win in ROLL_WINDOWS:
        row[f'target_roll{win}'] = np.mean(sm_history[-win:]) if len(sm_history) >= win else np.mean(sm_history)
    row['target_ewm7']  = pd.Series(sm_history[-50:]).ewm(span=7,  adjust=False).mean().iloc[-1]
    row['target_ewm14'] = pd.Series(sm_history[-60:]).ewm(span=14, adjust=False).mean().iloc[-1]

    X_row = np.array([row[c] for c in FEAT_COLS]).reshape(1, -1)

    # Weighted ensemble prediction
    pred = 0.5 * gbr.predict(X_row)[0] + 0.25 * etr.predict(X_row)[0] + 0.25 * rfr.predict(X_row)[0]
    test_predictions.append(pred)
    sm_history.append(pred)   # feed prediction back as next lag

print(f'Generated {len(test_predictions)} predictions')
print(f'Prediction range: {min(test_predictions):.4f} – {max(test_predictions):.4f}')


Generated 93 predictions
Prediction range: 0.1045 – 0.1481


# 6. Export Submission


In [11]:
submission = pd.DataFrame({
    'rowId': range(len(test_predictions)),
    'label': test_predictions
})
submission.to_csv(f'{base_path}/sample_submission.csv', index=False)
print('Submission saved to sample_submission.csv')
print(submission.head(10))


Submission saved to sample_submission.csv
   rowId     label
0      0  0.109821
1      1  0.110268
2      2  0.108762
3      3  0.108603
4      4  0.108554
5      5  0.107890
6      6  0.107325
7      7  0.106907
8      8  0.105940
9      9  0.104469
